# The model-selection atlas — every axis, one view

Across eighteen versions Onkos turned each silent modeling choice in the TGI→survival chain into an
explicit, quantified **model-selection axis**. This is the synthesis layer: a declarative **registry**
of the axes and a one-call **per-context survey** that runs each applicable axis and returns its native
headline — a map of *where the model-selection risk lies* for a context.

It is a survey, not a decomposition: the axes are not orthogonal and their headlines are in different
units, so the atlas flags `comparable = False` and points to `onkos.model_selection_budget` for the
rigorous orthogonal partition. *Population/trial level; inherits the context's tier; no recommendation.*

In [ ]:
import matplotlib
matplotlib.use('Agg')
import onkos
from onkos.atlas import AXES, model_selection_atlas

ds = onkos.load()
ctx = {'tumor_type': 'NSCLC', 'line': 'first'}

## The registry — the source of truth for the axes

`AXES` is the canonical list of the model-selection axes Onkos quantifies: what each varies, its
canonical finding, and the module + CLI for the deep dive.

In [ ]:
for ax in AXES:
    print(f'{ax.key:24s} [{ax.scope:11s} {ax.version}]  {ax.module}')
    print(f'{"":26s} varies: {ax.varies}')
    print(f'{"":26s} finding: {ax.finding}')

## The survey — where is the risk for this context?

One call runs each applicable single-agent axis and reports its native headline. The weeks-unit axes
(`os_swing_axes`) are a loosely-comparable leaderboard of 'weeks of median OS riding on this one
choice'; the detectability axes are in their own units.

In [ ]:
a = model_selection_atlas(ds, context=ctx)
print(f'tier {a.tier}   comparable={a.comparable}\n')
for e in a.entries:
    h = 'n/a' if e.headline is None else e.headline
    print(f'  {e.label:34s} {str(h):>6}  {e.unit}')
    print(f'  {"":34s}        - {e.detail}')
print()
print('OS-swing leaderboard (weeks), largest first:')
for e in sorted(a.os_swing_axes, key=lambda e: -(e.headline or 0)):
    print(f'  {e.label:34s} {e.headline:>6.0f} wk')

## The rigorous companion: the variance budget

The atlas surveys; it does not partition. For the orthogonal, additive decomposition of the OS-forecast
variance across the TGI-model and survival-link factors (+ parameter noise), use the budget — the
atlas's note points to it explicitly.

In [ ]:
print(a.note)
b = onkos.model_selection_budget(ds, context=ctx, n=80, seed=0)
print()
print(f'budget structural fraction = {b.structural_fraction:.2f}  (dominant axis: {b.dominant})')

## The takeaway

Eighteen versions of model-selection axes now have one entry point and one map. For NSCLC first line the
survival-structure and bridge-metric choices swing median OS the most (~100+ wk), the TGI-model choice
~40 wk, and the exposure-response shape ~20 wk on extrapolation — while 4 of 10 model pairs are
practically indistinguishable by a realistic trial. The atlas makes the project's central message
navigable: *here is every modeling choice that moves the forecast, and how much.* **No comparability
claim beyond the OS-swing group, no decomposition (see the budget), no individual prediction.**